# Курс «Продвинутое обучение с подкреплением» @ AI Masters
## Домашнее задание 6: Decision Transformer с памятью

### Введение

Decision Transformer (DT) — мощный алгоритм для оффлайн-обучения с подкреплением, обладающий значительной гибкостью для модификаций. Один из распространённых подходов — добавление механизмов памяти для улучшения его возможностей.

Хотя DT изначально способен справляться с частично наблюдаемыми задачами Маркова (POMDP) благодаря механизму внимания, внедрение явных структур памяти может помочь обрабатывать зависимости, выходящие за пределы типового контекста внимания.

В этом задании предлагается реализовать и оценить Decision Transformer с расширенной памятью для задач управления в POMDP.
> **Примечание:** Решения для GRU и LSTM будут добавлены после дедлайна.

### Установка

Перед началом установите необходимые зависимости:

In [ ]:
pip install torch numpy matplotlib gymnasium tqdm

### 1. POMDP-окружения

Используются классические задачи MDP из Gymnasium (CartPole, Pendulum, MountainCar), превращённые в POMDP с помощью специальных обёрток, которые вводят частичную наблюдаемость:

- `velocity_cartpole.py`: CartPole с скрытой информацией о скорости
- `flickering_pendulum.py`: Pendulum со случайно пропадающими наблюдениями
- `lidar_mountain_car.py`: MountainCar только с лидарами (датчиками расстояния)

**Важное замечание:** Код тщательно протестирован только с `VelocityCartPoleEnv`. Эксперименты с другими окружениями требуют дополнительных изменений:
- `FlickeringPendulumEnv`: Необходимо добавить поддержку непрерывных пространств действий
- `LiDARMountainCarEnv`: Необходимо реализовать методы сбора траекторий

В этом задании основной акцент на `VelocityCartPoleEnv`. Хотя Decision Transformer с длиной контекста > 1 должен решить эту задачу, цель — исследовать, как механизмы памяти могут её улучшить.

### 2. Сбор оффлайн-датасета RL

Поскольку мы работаем в оффлайн-режиме RL, сначала нужно собрать обучающие данные:

In [ ]:
python train_and_collect_data.py --env velocity_cartpole --train_timesteps 300000 --num_trajectories 100 --reward_threshold 475

Это обучит агента PPO-GRU, который затем будет использоваться для генерации траекторий.

После сбора датасета проверьте производительность агента PPO-GRU:

In [2]:
!python utils/visualize_ppo_agent.py --env velocity_cartpole --model_path pomdp_datasets/velocity_cartpole/recurrent_ppo_velocity_cartpole.pt --rnn_type gru

Visualization of velocity_cartpole from pomdp_datasets/velocity_cartpole/recurrent_ppo_velocity_cartpole.pt
RNN type: gru, Hidden dim: 128

Episode 1/5
Step 1: Action=1, Value=72.727
Step 2: Action=1, Value=72.785
Step 3: Action=1, Value=72.784
Step 4: Action=1, Value=72.787
Step 5: Action=1, Value=72.800
Step 6: Action=1, Value=72.804
Step 7: Action=0, Value=72.786
Step 8: Action=0, Value=72.535
Step 9: Action=0, Value=71.315
Step 10: Action=0, Value=70.285
Step 11: Action=0, Value=70.083
Step 12: Action=0, Value=70.083
Step 13: Action=0, Value=70.295
Step 14: Action=0, Value=70.927
Step 15: Action=0, Value=71.796
Step 16: Action=0, Value=72.449
Step 17: Action=0, Value=72.741
Step 18: Action=0, Value=72.831
Step 19: Action=0, Value=72.854
Step 20: Action=0, Value=72.860
Step 21: Action=0, Value=72.862
Step 22: Action=1, Value=72.863
Step 23: Action=1, Value=72.863
Step 24: Action=1, Value=72.862
Step 25: Action=1, Value=72.861
Step 26: Action=1, Value=72.858
Step 27: Action=1, Value=

И ознакомьтесь со статистикой датасета:

In [4]:
!python memory_dt.py --dataset pomdp_datasets/velocity_cartpole --stats_only

Analyzing 100 trajectories files in pomdp_datasets/velocity_cartpole
100%|███████████████████████████████████████| 100/100 [00:00<00:00, 4315.13it/s]

Dataset statistics:
Total episodes: 100
Total steps: 49712
Mean reward per episode: 497.12
Median reward per episode: 500.00
Min/Max reward: 477.00/500.00
Reward std: 6.66
Mean episode length: 497.12
Reward percentiles: 
  10%: 484.00
  25%: 500.00
  50%: 500.00
  75%: 500.00
  90%: 500.00
  95%: 500.00
  99%: 500.00
Saved histogram to plots/velocity_cartpole_rewards_histogram.png


Вы должны увидеть примерно такой вывод:

In [ ]:
Статистика датасета:
Всего эпизодов: 100
Всего шагов: 49924
Средняя награда за эпизод: 499.24
Медианная награда за эпизод: 500.00
Мин./Макс. награда: 476.00/500.00
Стандартное отклонение награды: 3.62
Средняя длина эпизода: 499.24
Процентили награды:
  10%: 500.00
  25%: 500.00
  50%: 500.00
  75%: 500.00
  90%: 500.00
  95%: 500.00
  99%: 500.00

Датасет содержит высококачественные траектории со средними наградами, близкими к максимуму для CartPole (500).

### 3. Decision Transformer с памятью

Сначала обучите и проверьте стандартный Decision Transformer в качестве базовой линии:

In [11]:
# Обучение базового DT
!python run_memory_dt.py --env velocity_cartpole --memory_type none --n_epochs 7 --eval_episodes 20


Traceback (most recent call last):
  File "/home/sergub/work/python/RL/HW/HW6/run_memory_dt.py", line 191, in <module>
    main() 
    ^^^^^^
  File "/home/sergub/work/python/RL/HW/HW6/run_memory_dt.py", line 106, in main
    dataset_path = os.path.join(args.data_dir, args.env)
                   ^^
UnboundLocalError: cannot access local variable 'os' where it is not associated with a value


In [7]:
# Проверка базового DT
!python utils/visualize_dt_agent.py --env velocity_cartpole --model_path models/memory_dt_velocity_cartpole_None_best.pt --memory_type none 

Model parameters: n_embed=64, n_layer=3, n_head=4, memory_dim=64
action_head structure: Linear
Visualizing velocity_cartpole agent from models/memory_dt_velocity_cartpole_None_best.pt
Memory type: none, Context length: 20

Episode 1/5
Target return: 500.0
Step 1: Initial action=0 (default)
Step 2: Action=0, RTG=499.00
Step 3: Action=0, RTG=498.00
Step 4: Action=0, RTG=497.00
Step 5: Action=0, RTG=496.00
Step 6: Action=1, RTG=495.00
Step 7: Action=1, RTG=494.00
Step 8: Action=1, RTG=493.00
Step 9: Action=1, RTG=492.00
Step 10: Action=1, RTG=491.00
Step 11: Action=1, RTG=490.00
Step 12: Action=1, RTG=489.00
Step 13: Action=1, RTG=488.00
Step 14: Action=1, RTG=487.00
Step 15: Action=1, RTG=486.00
Step 16: Action=1, RTG=485.00
Step 17: Action=1, RTG=484.00
Step 18: Action=1, RTG=483.00
Step 19: Action=1, RTG=482.00
Step 20: Action=0, RTG=481.00
Step 21: Action=0, RTG=480.00
Step 22: Action=0, RTG=479.00
Step 23: Action=0, RTG=478.00
Step 24: Action=0, RTG=477.00
Step 25: Action=0, RTG=476.

Сохраните результаты для дальнейшего сравнения с моделями, усиленными памятью.

### Задания

#### Задание 1: Реализация GRU и LSTM памяти (0.2 балла)

Рекуррентные нейронные сети, такие как LSTM и GRU, — естественный выбор для реализации памяти в нейронных сетях. Хотя Decision Transformer работает только с вниманием, добавление рекуррентной памяти — интересное направление.

Дополните код в `memory_dt.py` для реализации памяти на GRU и LSTM:

In [ ]:
# файл: memory_dt.py
self.pos_encoder = PositionalEncoding(n_embed)

# Память
if memory_type == 'gru':
   # TODO: Реализовать память на GRU
   self.memory_proj = nn.Linear(memory_dim, n_embed)
elif memory_type == 'lstm':
   # TODO: Реализовать память на LSTM
   self.memory_proj = nn.Linear(memory_dim, n_embed)
else:
   self.memory = None

# Transformer

In [ ]:
# файл: memory_dt.py
# добавить память
if self.memory is not None:
   if self.memory_type == 'gru':
         if self.hidden_state is None:
            # TODO: Реализовать память на GRU
         
         memory_out, self.hidden_state = self.memory(state_embeddings, self.hidden_state)
   elif self.memory_type == 'lstm':
         if self.hidden_state is None:
            # TODO: Реализовать память на LSTM
         
         memory_out, self.hidden_state = self.memory(state_embeddings, self.hidden_state)
   
   # проекция памяти в размерность эмбеддинга
   memory_embedding = self.memory_proj(memory_out)

После реализации модулей памяти обучите улучшенные модели:

In [7]:
# Обучение DT+GRU (0.1 балла)
!python run_memory_dt.py --env velocity_cartpole --memory_type gru --n_epochs 7 --eval_episodes 20

Training Memory Decision Transformer on velocity_cartpole...
Dataset stats: total=49652, train=44686, val=4966, state_dim=2, actions=2
Epoch 1/7 [Train]: 100%|█| 699/699 [00:10<00:00, 66.31it/s, loss=0.3406, avg_los
Epoch 1/7: Train Loss=0.4606, Val Loss=0.3730
Running environment validation...
Episode 1: Return=269.0, Steps=269
Episode 2: Return=500.0, Steps=500
Episode 3: Return=328.0, Steps=328
Episode 4: Return=500.0, Steps=500
Episode 5: Return=320.0, Steps=320
Episode 6: Return=240.0, Steps=240
Episode 7: Return=500.0, Steps=500
Episode 8: Return=478.0, Steps=478
Episode 9: Return=179.0, Steps=179
Episode 10: Return=500.0, Steps=500
Validation: Mean Return=381.40, Success Rate=50.00%
New best model with return 381.40
Epoch 2/7 [Train]: 100%|█| 699/699 [00:10<00:00, 68.24it/s, loss=0.3644, avg_los
Epoch 2/7: Train Loss=0.3457, Val Loss=0.3384
Running environment validation...
Episode 1: Return=500.0, Steps=500
Episode 2: Return=500.0, Steps=500
Episode 3: Return=500.0, Steps=500
E

In [8]:
# Обучение DT+LSTM (0.1 балла)
!python run_memory_dt.py --env velocity_cartpole --memory_type lstm --n_epochs 7 --eval_episodes 20

Training Memory Decision Transformer on velocity_cartpole...
Dataset stats: total=49652, train=44686, val=4966, state_dim=2, actions=2
Using LSTM memory with 64 hidden units
Epoch 1/7 [Train]:  45%|▍| 316/699 [00:04<00:05, 65.76it/s, loss=0.4341, avg_los
Traceback (most recent call last):
  File "/home/sergub/work/python/RL/HW/HW6/run_memory_dt.py", line 159, in <module>
    main() 
    ^^^^^^
  File "/home/sergub/work/python/RL/HW/HW6/run_memory_dt.py", line 125, in main
    model, train_losses, val_returns = train_memory_dt(
                                       ^^^^^^^^^^^^^^^^
  File "/home/sergub/work/python/RL/HW/HW6/memory_dt.py", line 419, in train_memory_dt
    loss.backward()
  File "/home/sergub/work/python/venvs/env1/lib/python3.12/site-packages/torch/_tensor.py", line 626, in backward
    torch.autograd.backward(
  File "/home/sergub/work/python/venvs/env1/lib/python3.12/site-packages/torch/autograd/__init__.py", line 347, in backward
    _engine_run_backward(
  File "/ho

Проверьте их производительность:

In [9]:
# Проверка DT+GRU
!python utils/visualize_dt_agent.py --env velocity_cartpole --model_path models/memory_dt_velocity_cartpole_gru_best.pt --memory_type gru 

Model parameters: n_embed=64, n_layer=3, n_head=4, memory_dim=64
action_head structure: Linear
Using GRU memory with 64 hidden units
Visualizing velocity_cartpole agent from models/memory_dt_velocity_cartpole_gru_best.pt
Memory type: gru, Context length: 20

Episode 1/5
Target return: 500.0
Step 1: Initial action=0 (default)
Step 2: Action=1, RTG=499.00
Step 3: Action=1, RTG=498.00
Step 4: Action=1, RTG=497.00
Step 5: Action=0, RTG=496.00
Step 6: Action=0, RTG=495.00
Step 7: Action=1, RTG=494.00
Step 8: Action=0, RTG=493.00
Step 9: Action=0, RTG=492.00
Step 10: Action=1, RTG=491.00
Step 11: Action=1, RTG=490.00
Step 12: Action=1, RTG=489.00
Step 13: Action=0, RTG=488.00
Step 14: Action=0, RTG=487.00
Step 15: Action=0, RTG=486.00
Step 16: Action=0, RTG=485.00
Step 17: Action=1, RTG=484.00
Step 18: Action=1, RTG=483.00
Step 19: Action=1, RTG=482.00
Step 20: Action=1, RTG=481.00
Step 21: Action=0, RTG=480.00
Step 22: Action=0, RTG=479.00
Step 23: Action=0, RTG=478.00
Step 24: Action=0, RT

In [11]:
# Проверка DT+LSTM
!python utils/visualize_dt_agent.py --env velocity_cartpole --model_path models/memory_dt_velocity_cartpole_lstm_best.pt --memory_type lstm 

Model parameters: n_embed=64, n_layer=3, n_head=4, memory_dim=64
action_head structure: Linear
Using LSTM memory
Visualizing velocity_cartpole agent from models/memory_dt_velocity_cartpole_lstm_best.pt
Memory type: lstm, Context length: 20

Episode 1/5
Target return: 500.0
Step 1: Initial action=0 (default)
Step 2: Action=1, RTG=499.00
Step 3: Action=1, RTG=498.00
Step 4: Action=1, RTG=497.00
Step 5: Action=1, RTG=496.00
Step 6: Action=1, RTG=495.00
Step 7: Action=1, RTG=494.00
Step 8: Action=1, RTG=493.00
Step 9: Action=0, RTG=492.00
Step 10: Action=0, RTG=491.00
Step 11: Action=0, RTG=490.00
Step 12: Action=0, RTG=489.00
Step 13: Action=0, RTG=488.00
Step 14: Action=0, RTG=487.00
Step 15: Action=0, RTG=486.00
Step 16: Action=0, RTG=485.00
Step 17: Action=0, RTG=484.00
Step 18: Action=0, RTG=483.00
Step 19: Action=0, RTG=482.00
Step 20: Action=0, RTG=481.00
Step 21: Action=0, RTG=480.00
Step 22: Action=1, RTG=479.00
Step 23: Action=1, RTG=478.00
Step 24: Action=1, RTG=477.00
Step 25: 

#### Задание 2: Сравнительный анализ (0.3 балла)

Сравните производительность стандартного DT с реализациями DT+GRU и DT+LSTM. Проанализируйте:

- Эффективность обучения (скорость сходимости)
- Итоговые метрики производительности
- Объём данных, необходимый для сходимости
- Общий эффект добавления модулей памяти

Проведите эксперименты с различными параметрами моделей (длина контекста, размер обучающей выборки, качество данных и др.) и сравните результаты. Что происходит при изменении return-to-go (RTG)?

Приложите графики обучения и метрики для подтверждения анализа.

#### Задание 3: Собственный механизм памяти (0.3 балла)

Разработайте и реализуйте собственный механизм памяти для Decision Transformer. Здесь нет единственно правильного ответа — проявите креативность в разумных пределах.

Опишите ваш подход:
- Какую информацию будет обрабатывать ваш механизм?
- Как информация будет обрабатываться?
- Какую архитектуру модели вы используете?
- Какой подход к обучению выбран?

Проведите эксперименты и сравните вашу реализацию с предыдущими подходами. Какие выводы можно сделать?

In [5]:
!python run_memory_dt.py --env velocity_cartpole --memory_type mlp_vector --n_epochs 7 --eval_episodes 20

Training Memory Decision Transformer on velocity_cartpole...
Dataset stats: total=49652, train=44686, val=4966, state_dim=2, actions=2
Using MLP memory with 64 hidden units
Epoch 1/7 [Train]: 100%|█| 699/699 [00:13<00:00, 51.29it/s, loss=0.3477, avg_los
Epoch 1/7: Train Loss=0.4812, Val Loss=0.3641
Running environment validation...
Episode 1: Return=51.0, Steps=51
Episode 2: Return=11.0, Steps=11
Episode 3: Return=63.0, Steps=63
Episode 4: Return=57.0, Steps=57
Episode 5: Return=83.0, Steps=83
Episode 6: Return=74.0, Steps=74
Episode 7: Return=12.0, Steps=12
Episode 8: Return=74.0, Steps=74
Episode 9: Return=55.0, Steps=55
Episode 10: Return=63.0, Steps=63
Validation: Mean Return=54.30, Success Rate=0.00%
New best model with return 54.30
Epoch 2/7 [Train]: 100%|█| 699/699 [00:13<00:00, 52.09it/s, loss=0.3606, avg_los
Epoch 2/7: Train Loss=0.3555, Val Loss=0.3747
Running environment validation...
Episode 1: Return=15.0, Steps=15
Episode 2: Return=14.0, Steps=14
Episode 3: Return=14.0, S

In [6]:
!python utils/visualize_dt_agent.py --env velocity_cartpole --model_path models/memory_dt_velocity_cartpole_mlp_vector_best.pt --memory_type mlp_vector 

Model parameters: n_embed=64, n_layer=3, n_head=4, memory_dim=64
action_head structure: Linear
Using MLP memory with 64 hidden units
Visualizing velocity_cartpole agent from models/memory_dt_velocity_cartpole_mlp_vector_best.pt
Memory type: mlp_vector, Context length: 20

Episode 1/5
Target return: 500.0
Step 1: Initial action=0 (default)
Step 2: Action=1, RTG=499.00
Step 3: Action=1, RTG=498.00
Step 4: Action=1, RTG=497.00
Step 5: Action=1, RTG=496.00
Step 6: Action=1, RTG=495.00
Step 7: Action=1, RTG=494.00
Step 8: Action=1, RTG=493.00
Step 9: Action=0, RTG=492.00
Step 10: Action=0, RTG=491.00
Step 11: Action=0, RTG=490.00
Step 12: Action=0, RTG=489.00
Step 13: Action=0, RTG=488.00
Step 14: Action=0, RTG=487.00
Step 15: Action=0, RTG=486.00
Step 16: Action=0, RTG=485.00
Step 17: Action=0, RTG=484.00
Step 18: Action=0, RTG=483.00
Step 19: Action=0, RTG=482.00
Step 20: Action=0, RTG=481.00
Step 21: Action=0, RTG=480.00
Step 22: Action=0, RTG=479.00
Step 23: Action=0, RTG=478.00
Step 24

### Правила сдачи (0.2 балла)

1. Форкните или склонируйте этот репозиторий и внесите изменения
2. Добавьте отчёт с результатами экспериментов и анализом в любом удобном формате (.ipynb, .pdf и т.д.) в репозиторий
3. Отправьте файл со ссылкой на ваш репозиторий в телеграм-бот для сдачи

Хорошо структурированный, профессионально оформленный отчёт приносит 0.2 балла.

### Структура репозитория

- `pomdp_envs/velocity_cartpole.py`: Реализация CartPole с сокрытием скорости
- `pomdp_envs/flickering_pendulum.py`: Реализация Pendulum с мерцающими наблюдениями
- `pomdp_envs/lidar_mountain_car.py`: Реализация MountainCar с лидарами
- `recurrent_ppo.py`: Реализация рекуррентного PPO для сбора данных
- `train_and_collect_data.py`: Скрипт обучения агента PPO и сбора траекторий
- `memory_dt.py`: Реализация Decision Transformer с памятью
- `run_memory_dt.py`: Скрипт для обучения и оценки Decision Transformer с памятью



---

Если нужен экспорт в другой формат или разметка для другого инструмента — уточните, пожалуйста!